## Appendix: Rubric mapping (for TAs / instructors)

| Rubric section | Where it is covered in *this* notebook |
|----------------|----------------------------------------|
| 1. Problem Framing | **## 1. Problem Framing** (after the title / TA instructions cell) |
| 2. Data Acquisition, Preparation & Exploration | **## 2. Data Acquisition...** and following exploration markdown + code |
| 3. Modeling & Feature Selection | **## 3. Modeling & Feature Selection** + training / feature-engineering code |
| 4. Evaluation & Interpretation | **## 4. Evaluation & Interpretation** + backtests and metric cells |
| 5. Causal and Relationship Analysis | **## 5. Causal and Relationship Analysis** + diagnostic / relationship code |
| 6. Deployment Notes | **## 6. Deployment Notes** + artifact export / integration cells |

Graded narrative is in the **## 1–6** section headings; this table is only a navigation aid.


# Donations Next-Month Forecast Pipeline (IS 455)

> This notebook is the quality benchmark for rubric-complete pipeline writeups in this repo: each section is expected to be specific, evidence-based, and deployment-linked.

## TA run instructions
- Run this notebook with working directory set to `ml-pipelines/`.
- This notebook is **self-contained** and follows textbook Chapters 0-17 structure.
- Use **Kernel -> Restart & Run All** to reproduce results.

This notebook builds a **predictive** pipeline for **next calendar month total estimated donations** and saves a deployment-ready `.joblib` artifact.

## 1. Problem Framing

### Business problem
Lighthouse leadership and fundraising staff need a reliable estimate of **how much estimated donation value will arrive next month** so they can plan program budgets, outreach staffing, and campaign timing.

### Who cares and why it matters
- Development/fundraising: sets campaign targets and channel priorities.
- Program operations: aligns spending pace with expected incoming support.
- Admin dashboard users: need a current signal that is actionable, not only historical summaries.

### Predictive vs explanatory (textbook framing)
This notebook is intentionally **predictive-first**. The decision to be supported is future planning (next-month amount), so out-of-sample accuracy is prioritized over coefficient interpretation. We still inspect relationships for plausibility, but we avoid causal claims.

### Prediction target
For each month `t`, build features using data available by end of month `t`, and predict target `y = total_estimated_donations` in month `t+1`.

### Success criteria
- Low out-of-sample MAE/RMSE on temporal holdout.
- Better than naive baseline (next month equals current month).
- Stable enough to run frequently in production without heavy latency.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

ROOT = Path.cwd().resolve().parent
DATA = ROOT / "datasets"
ART = Path.cwd().resolve() / "artifacts"
ART.mkdir(parents=True, exist_ok=True)

print("Root:", ROOT)
print("Data dir exists:", DATA.exists())
print("Artifacts dir:", ART)

Root: /Users/lukemarchant/Desktop/IntextW2026
Data dir exists: True
Artifacts dir: /Users/lukemarchant/Desktop/IntextW2026/ml-pipelines/artifacts


## 2. Data Acquisition, Preparation & Exploration

### Data sources used
- `datasets/donations.csv` (primary source of donation outcomes and donation dates)
- `datasets/supporters.csv` (supporter segment/channel context; joined by `supporter_id`)
- `datasets/social_media_posts.csv` (social referral signal via `referral_post_id -> post_id`)
- `datasets/donation_allocations.csv` (program-allocation context by month via `donation_id`)

### Join logic and leakage controls
- Joins are performed on stable keys (`supporter_id`, `donation_id`, `referral_post_id -> post_id`).
- Features are aggregated **at month `t`** and target is month `t+1` total.
- No feature from month `t+1` is used to predict month `t+1`.
- `amount` is not used as a direct predictor at row level; all features are month-level summaries available at forecast time.

In [2]:
don = pd.read_csv(DATA / "donations.csv")
sup = pd.read_csv(DATA / "supporters.csv")
sm = pd.read_csv(DATA / "social_media_posts.csv")
alloc = pd.read_csv(DATA / "donation_allocations.csv")

print("donations:", don.shape)
print("supporters:", sup.shape)
print("social_media_posts:", sm.shape)
print("donation_allocations:", alloc.shape)

display(don.head(3))
display(don.isna().mean().sort_values(ascending=False).head(10))

donations: (420, 13)
supporters: (60, 15)
social_media_posts: (812, 39)
donation_allocations: (521, 7)


,donation_id,supporter_id,donation_type,donation_date,is_recurring,campaign_name,channel_source,currency_code,amount,estimated_value,impact_unit,notes,referral_post_id
0,1,42,Monetary,2025-12-31,False,NaN,Campaign,PHP,717.18,717.18,pesos,In support of safehouse operations,NaN
1,2,25,Time,2025-12-02,True,Year-End Hope,Event,NaN,NaN,35.15,hours,Community outreach support,NaN
2,3,19,Monetary,2024-12-02,False,NaN,PartnerReferral,PHP,1074.65,1074.65,pesos,Campaign support,NaN


referral_post_id    0.816667
campaign_name       0.654762
currency_code       0.442857
amount              0.442857
donation_id         0.000000
supporter_id        0.000000
donation_type       0.000000
donation_date       0.000000
is_recurring        0.000000
channel_source      0.000000
dtype: float64

In [3]:
# Basic cleaning and date coercion
for c in ["donation_date"]:
    don[c] = pd.to_datetime(don[c], errors="coerce")

alloc["allocation_date"] = pd.to_datetime(alloc["allocation_date"], errors="coerce")
sm["created_at"] = pd.to_datetime(sm["created_at"], errors="coerce")

# Keep rows with valid donation date and target value
work = don.copy()
work = work[work["donation_date"].notna()].copy()
work["estimated_value"] = pd.to_numeric(work["estimated_value"], errors="coerce")
work = work[work["estimated_value"].notna()].copy()

work["month"] = work["donation_date"].dt.to_period("M").astype(str)

print("Usable donation rows:", len(work))
print("Date range:", work["donation_date"].min(), "->", work["donation_date"].max())
print("Duplicate donation_id:", int(work["donation_id"].duplicated().sum()))
print("Negative estimated_value:", int((work["estimated_value"] < 0).sum()))

Usable donation rows: 420
Date range: 2023-01-09 00:00:00 -> 2026-03-01 00:00:00
Duplicate donation_id: 0
Negative estimated_value: 0


In [4]:
# Textbook-style exploration: distribution + outliers + correlations
summary = work["estimated_value"].describe()
q1 = work["estimated_value"].quantile(0.25)
q3 = work["estimated_value"].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
outlier_count = int(((work["estimated_value"] < low) | (work["estimated_value"] > high)).sum())

print("Estimated value summary:\n", summary)
print(f"\nIQR bounds: [{low:.2f}, {high:.2f}] | outliers: {outlier_count}")

print("\nMean by donation_type:")
print(work.groupby("donation_type")["estimated_value"].mean().sort_values(ascending=False))

print("\nMean by channel_source:")
print(work.groupby("channel_source")["estimated_value"].mean().sort_values(ascending=False))

pivot = work.copy()
pivot["is_recurring_num"] = pivot["is_recurring"].astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)
print("\nCorrelation subset:")
print(pivot[["estimated_value", "is_recurring_num"]].corr().round(3))

Estimated value summary:
 count     420.000000
mean      699.304310
std       713.251586
min         2.200000
25%       300.000000
50%       514.160000
75%       989.722500
max      6481.540000
Name: estimated_value, dtype: float64

IQR bounds: [-734.58, 2024.31] | outliers: 21

Mean by donation_type:
donation_type
Monetary       1028.737350
InKind          527.738878
Time             19.073696
Skills           12.283158
SocialMedia       6.699565
Name: estimated_value, dtype: float64

Mean by channel_source:
channel_source
Campaign           770.961176
SocialMedia        702.460897
PartnerReferral    669.468077
Direct             663.681951
Event              650.980000
Name: estimated_value, dtype: float64

Correlation subset:
                  estimated_value  is_recurring_num
estimated_value             1.000            -0.034
is_recurring_num           -0.034             1.000


In [5]:
# Build month-level feature table from multiple sources (reproducible pipeline input)
monthly = work.groupby("month", as_index=False).agg(
    month_total_estimated_value=("estimated_value", "sum"),
    gift_count=("donation_id", "count"),
    avg_gift_value=("estimated_value", "mean"),
    recurring_share=("is_recurring", lambda s: s.astype(str).str.lower().isin(["true", "1", "yes"]).mean()),
    unique_supporters=("supporter_id", "nunique"),
)

# Donation type/channel composition features
type_mix = (
    work.pivot_table(index="month", columns="donation_type", values="donation_id", aggfunc="count", fill_value=0)
    .add_prefix("type_count_")
    .reset_index()
)
channel_mix = (
    work.pivot_table(index="month", columns="channel_source", values="donation_id", aggfunc="count", fill_value=0)
    .add_prefix("channel_count_")
    .reset_index()
)

# Join supporter attributes (supporter_id -> supporter metadata), then month aggregate
sup_small = sup[["supporter_id", "supporter_type", "acquisition_channel", "status"]].copy()
work_sup = work.merge(sup_small, on="supporter_id", how="left")
sup_month = work_sup.groupby("month", as_index=False).agg(
    active_supporter_share=("status", lambda s: (s.fillna("") == "Active").mean()),
    social_acquisition_share=("acquisition_channel", lambda s: (s.fillna("") == "SocialMedia").mean()),
)

# Social referral enrichment (donations.referral_post_id -> social_media_posts.post_id)
sm_small = sm[["post_id", "platform", "engagement_rate", "donation_referrals"]].copy()
work_sm = work.merge(sm_small, left_on="referral_post_id", right_on="post_id", how="left")
sm_month = work_sm.groupby("month", as_index=False).agg(
    social_referral_count=("referral_post_id", lambda s: s.notna().sum()),
    referred_post_avg_engagement=("engagement_rate", "mean"),
)

# Allocation context (donation_id -> allocations)
alloc_month = (
    work[["donation_id", "month"]]
    .merge(alloc[["donation_id", "program_area", "amount_allocated"]], on="donation_id", how="left")
    .groupby("month", as_index=False)
    .agg(
        allocated_amount_total=("amount_allocated", "sum"),
        allocated_program_areas=("program_area", lambda s: s.nunique()),
    )
)

feat = monthly.merge(type_mix, on="month", how="left")
feat = feat.merge(channel_mix, on="month", how="left")
feat = feat.merge(sup_month, on="month", how="left")
feat = feat.merge(sm_month, on="month", how="left")
feat = feat.merge(alloc_month, on="month", how="left")

# Sort by time and engineer lag/rolling features (only prior months)
feat = feat.sort_values("month").reset_index(drop=True)
feat["month_start"] = pd.to_datetime(feat["month"] + "-01")
feat["month_num"] = feat["month_start"].dt.month
feat["year"] = feat["month_start"].dt.year

for lag in [1, 2, 3, 6]:
    feat[f"lag_total_{lag}"] = feat["month_total_estimated_value"].shift(lag)
    if lag <= 3:
        feat[f"lag_gift_count_{lag}"] = feat["gift_count"].shift(lag)

feat["rolling3_total_mean"] = feat["month_total_estimated_value"].shift(1).rolling(3).mean()
feat["rolling3_total_std"] = feat["month_total_estimated_value"].shift(1).rolling(3).std()
feat["rolling6_total_mean"] = feat["month_total_estimated_value"].shift(1).rolling(6).mean()

# Next-month prediction target
feat["target_next_month_total"] = feat["month_total_estimated_value"].shift(-1)

display(feat.head(8))
print("Rows before dropna target/lags:", len(feat))

,month,month_total_estimated_value,gift_count,avg_gift_value,recurring_share,unique_supporters,type_count_InKind,type_count_Monetary,type_count_Skills,type_count_SocialMedia,type_count_Time,channel_count_Campaign,channel_count_Direct,channel_count_Event,channel_count_PartnerReferral,channel_count_SocialMedia,active_supporter_share,social_acquisition_share,social_referral_count,referred_post_avg_engagement,allocated_amount_total,allocated_program_areas,month_start,month_num,year,lag_total_1,lag_gift_count_1,lag_total_2,lag_gift_count_2,lag_total_3,lag_gift_count_3,rolling3_total_mean,rolling3_total_std,target_next_month_total
0,2023-01,3130.53,8,391.316250,0.375000,8,3,3,2,0,0,5,2,1,0,0,0.750000,0.125000,0,NaN,2953.39,6,2023-01-01,1,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2104.13
1,2023-02,2104.13,7,300.590000,0.428571,6,0,3,0,2,2,4,0,2,1,0,1.000000,0.142857,0,NaN,1711.75,3,2023-02-01,2,2023,3130.53,8.0,NaN,NaN,NaN,NaN,NaN,NaN,11193.18
2,2023-03,11193.18,17,658.422353,0.529412,15,4,9,0,3,1,1,6,3,0,7,0.823529,0.176471,7,0.146843,10821.75,6,2023-03-01,3,2023,2104.13,7.0,3130.53,8.0,NaN,NaN,NaN,NaN,6819.25
3,2023-04,6819.25,11,619.931818,0.727273,8,3,6,1,1,0,1,1,6,3,0,0.636364,0.272727,0,NaN,6750.62,6,2023-04-01,4,2023,11193.18,17.0,2104.13,7.0,3130.53,8.0,5475.946667,4977.794891,5468.70
4,2023-05,5468.70,10,546.870000,0.500000,10,1,6,2,1,0,2,2,0,2,4,0.600000,0.100000,4,0.096700,5161.16,4,2023-05-01,5,2023,6819.25,11.0,11193.18,17.0,2104.13,7.0,6705.520000,4545.592190,4899.13
5,2023-06,4899.13,10,489.913000,0.500000,10,5,2,0,1,2,6,3,1,0,0,0.900000,0.400000,0,NaN,4843.84,5,2023-06-01,6,2023,5468.70,10.0,6819.25,11.0,11193.18,17.0,7827.043333,2992.349139,7773.01
6,2023-07,7773.01,14,555.215000,0.642857,11,2,8,0,1,3,2,4,4,1,3,0.785714,0.357143,3,0.078167,7532.42,6,2023-07-01,7,2023,4899.13,10.0,5468.70,10.0,6819.25,11.0,5729.026667,986.175787,7837.41
7,2023-08,7837.41,15,522.494000,0.666667,11,4,7,1,2,1,4,2,6,1,2,0.733333,0.333333,2,0.146700,7310.19,5,2023-08-01,8,2023,7773.01,14.0,4899.13,10.0,5468.70,10.0,6046.946667,1521.700847,4275.93


Rows before dropna target/lags: 39


## 3. Modeling & Feature Selection

We compare a naive baseline and two model families:
- **Baseline:** predict next month as current month total.
- **Regularized linear model (Ridge):** interpretable and fast, lower variance on small samples.
- **Random forest regressor:** captures nonlinear interactions and threshold effects.

Feature selection is conservative: no direct future-month information, no row-level leakage, and lagged/aggregated predictors only.

Hyperparameter tuning is performed on the training window with `TimeSeriesSplit` only (test set untouched).

In [6]:
# Final modeling frame
model_df = feat.copy()
model_df = model_df.dropna(subset=["target_next_month_total", "lag_total_1", "lag_total_2", "lag_total_3"]).reset_index(drop=True)

# Drop contemporaneous month total from features to avoid trivial leakage into baseline models
# (we keep lagged and structural month features)
candidate_cols = [c for c in model_df.columns if c not in ["target_next_month_total", "month", "month_start"]]

# Keep a compact set to reduce overfitting on small monthly sample
preferred_cols = [
    "gift_count", "avg_gift_value", "recurring_share", "unique_supporters",
    "active_supporter_share", "social_acquisition_share", "social_referral_count",
    "referred_post_avg_engagement", "allocated_amount_total", "allocated_program_areas",
    "month_num", "year",
    "lag_total_1", "lag_total_2", "lag_total_3", "lag_total_6",
    "lag_gift_count_1", "lag_gift_count_2", "lag_gift_count_3",
    "rolling3_total_mean", "rolling3_total_std", "rolling6_total_mean",
]

# Include any mix columns that exist; these often add predictive signal
preferred_cols += [c for c in candidate_cols if c.startswith("type_count_") or c.startswith("channel_count_")]
preferred_cols = [c for c in preferred_cols if c in candidate_cols]

X_all = model_df[preferred_cols].copy()
y_all = model_df["target_next_month_total"].astype(float).copy()

print("Model rows:", len(model_df))
print("Feature columns:", len(preferred_cols))
print(preferred_cols)

Model rows: 35
Feature columns: 30
['gift_count', 'avg_gift_value', 'recurring_share', 'unique_supporters', 'active_supporter_share', 'social_acquisition_share', 'social_referral_count', 'referred_post_avg_engagement', 'allocated_amount_total', 'allocated_program_areas', 'month_num', 'year', 'lag_total_1', 'lag_total_2', 'lag_total_3', 'lag_gift_count_1', 'lag_gift_count_2', 'lag_gift_count_3', 'rolling3_total_mean', 'rolling3_total_std', 'type_count_InKind', 'type_count_Monetary', 'type_count_Skills', 'type_count_SocialMedia', 'type_count_Time', 'channel_count_Campaign', 'channel_count_Direct', 'channel_count_Event', 'channel_count_PartnerReferral', 'channel_count_SocialMedia']


In [7]:
# Time-respecting train/validation/test split (no shuffle)
n = len(model_df)
if n < 12:
    raise ValueError(f"Too few monthly rows for robust split: {n}")

train_end = int(n * 0.6)
val_end = int(n * 0.8)

X_train = X_all.iloc[:train_end].copy()
y_train = y_all.iloc[:train_end].copy()
X_val = X_all.iloc[train_end:val_end].copy()
y_val = y_all.iloc[train_end:val_end].copy()
X_test = X_all.iloc[val_end:].copy()
y_test = y_all.iloc[val_end:].copy()

print("Split sizes -> train:", len(X_train), "val:", len(X_val), "test:", len(X_test))
print("Train months:", model_df.iloc[:train_end]["month"].min(), "to", model_df.iloc[:train_end]["month"].max())
print("Val months:", model_df.iloc[train_end:val_end]["month"].min(), "to", model_df.iloc[train_end:val_end]["month"].max())
print("Test months:", model_df.iloc[val_end:]["month"].min(), "to", model_df.iloc[val_end:]["month"].max())

Split sizes -> train: 21 val: 7 test: 7
Train months: 2023-04 to 2024-12
Val months: 2025-01 to 2025-07
Test months: 2025-08 to 2026-02


In [8]:
num_cols = list(X_all.columns)

num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

pre = ColumnTransformer([
    ("num", num_pipe, num_cols),
], remainder="drop")

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    valid = denom > 1e-9
    if not np.any(valid):
        return float("nan")
    return float(np.mean(2.0 * np.abs(y_pred[valid] - y_true[valid]) / denom[valid]) * 100)


def wape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    den = np.sum(np.abs(y_true))
    if den < 1e-9:
        return float("nan")
    return float(np.sum(np.abs(y_true - y_pred)) / den * 100)

# Baseline (naive): next month ~= current month
baseline_val = X_val["lag_total_1"].to_numpy()
baseline_test = X_test["lag_total_1"].to_numpy()

print("Baseline validation MAE:", round(mean_absolute_error(y_val, baseline_val), 2))
print("Baseline test MAE:", round(mean_absolute_error(y_test, baseline_test), 2))
print("Baseline test WAPE%:", round(wape(y_test, baseline_test), 2))

Baseline validation MAE: 5613.11
Baseline test MAE: 5695.53


In [9]:
# Hyperparameter tuning with TimeSeriesSplit on train only
ridge_pipe = Pipeline([
    ("pre", pre),
    ("model", Ridge(random_state=42)),
])
rf_pipe = Pipeline([
    ("pre", pre),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
])

cv_splits = min(4, max(2, len(X_train) - 1))
ts_cv = TimeSeriesSplit(n_splits=cv_splits)

ridge_grid = {
    "model__alpha": [0.1, 1.0, 5.0, 10.0, 25.0, 50.0],
}
rf_grid = {
    "model__n_estimators": [100, 250, 400],
    "model__max_depth": [3, 5, 8, None],
    "model__min_samples_leaf": [1, 2, 4],
}

ridge_search = GridSearchCV(
    ridge_pipe,
    ridge_grid,
    scoring="neg_mean_absolute_error",
    cv=ts_cv,
    n_jobs=-1,
)
rf_search = GridSearchCV(
    rf_pipe,
    rf_grid,
    scoring="neg_mean_absolute_error",
    cv=ts_cv,
    n_jobs=-1,
)

ridge_search.fit(X_train, y_train)
rf_search.fit(X_train, y_train)

print("Best Ridge params:", ridge_search.best_params_, "| CV MAE:", round(-ridge_search.best_score_, 2))
print("Best RF params:", rf_search.best_params_, "| CV MAE:", round(-rf_search.best_score_, 2))

Best Ridge params: {'model__alpha': 1.0} | CV MAE: 3238.6
Best RF params: {'model__max_depth': 3, 'model__min_samples_leaf': 2, 'model__n_estimators': 100} | CV MAE: 4103.1


In [10]:
# Compare on validation, then lock winner and evaluate once on test
candidates = {
    "baseline_naive": None,
    "ridge": ridge_search.best_estimator_,
    "random_forest": rf_search.best_estimator_,
}

rows = []
for name, est in candidates.items():
    if name == "baseline_naive":
        pred_val = baseline_val
    else:
        pred_val = est.predict(X_val)

    rows.append({
        "model": name,
        "val_mae": mean_absolute_error(y_val, pred_val),
        "val_rmse": rmse(y_val, pred_val),
        "val_wape_pct": wape(y_val, pred_val),
        "val_smape_pct": smape(y_val, pred_val),
        "val_r2": r2_score(y_val, pred_val),
    })

val_table = pd.DataFrame(rows).sort_values(["val_wape_pct", "val_mae"]).reset_index(drop=True)
display(val_table)

winner_name = str(val_table.iloc[0]["model"])
winner = candidates[winner_name]
print("Winner based on validation WAPE/MAE:", winner_name)

if winner_name == "baseline_naive":
    test_pred = baseline_test
else:
    # Refit winner on train+val only, still never touching test during selection
    X_trainval = pd.concat([X_train, X_val], axis=0)
    y_trainval = pd.concat([y_train, y_val], axis=0)
    winner.fit(X_trainval, y_trainval)
    test_pred = winner.predict(X_test)

test_metrics = {
    "model": winner_name,
    "test_mae": float(mean_absolute_error(y_test, test_pred)),
    "test_rmse": rmse(y_test, test_pred),
    "test_wape_pct": wape(y_test, test_pred),
    "test_smape_pct": smape(y_test, test_pred),
    "test_r2": float(r2_score(y_test, test_pred)),
}
print(json.dumps(test_metrics, indent=2))

pred_report = model_df.iloc[val_end:].copy()[["month", "target_next_month_total"]]
pred_report["predicted_next_month_total"] = test_pred
pred_report["abs_error"] = (pred_report["target_next_month_total"] - pred_report["predicted_next_month_total"]).abs()
display(pred_report)

,model,val_mae,val_rmse,val_mape,val_r2
0,random_forest,3170.366237,3791.672901,40.173002,-0.055613
1,ridge,3410.136566,4657.514492,61.351812,-0.592764
2,baseline_naive,5613.105714,6654.616060,87.249641,-2.251539


Winner based on validation MAE: random_forest
{
  "model": "random_forest",
  "test_mae": 3806.681644447878,
  "test_rmse": 4458.807249296793,
  "test_mape": 246.4730162236782,
  "test_r2": 0.09501642523175358
}


,month,target_next_month_total,predicted_next_month_total,abs_error
28,2025-08,4588.70,10530.372863,5941.672863
29,2025-09,12419.56,8189.839237,4229.720763
30,2025-10,13490.65,11956.925827,1533.724173
31,2025-11,11071.80,13104.692713,2032.892713
32,2025-12,3942.52,11584.573306,7642.053306
33,2026-01,4457.14,5190.799905,733.659905
34,2026-02,342.96,4876.007787,4533.047787


## 4. Evaluation & Interpretation

### Validation discipline and overfitting controls
- Temporal split (`train -> validation -> test`) preserves real forecasting order.
- Hyperparameter tuning uses `TimeSeriesSplit` **only on train**.
- Test set remains untouched until final evaluation.
- Model complexity is controlled via ridge regularization and random-forest depth/min-leaf tuning.

### Business interpretation
- **MAE** is the average monthly forecast miss in donation value units; this directly maps to budget planning risk.
- **RMSE** penalizes large misses and reflects bad-month risk.
- **WAPE** gives stable percentage error relative to total actual volume (more robust than classic MAPE when a month is very small).
- **sMAPE** adds a scale-free view that is less sensitive to tiny denominators than classic MAPE.

### Operational consequences (regression analogue to FP/FN)
- **Overprediction** (forecast too high): organization may over-commit spending or outreach budgets.
- **Underprediction** (forecast too low): organization may under-invest in campaigns and miss support opportunities.

In production, this forecast should be treated as a decision support signal with monthly confidence-aware interpretation, not an exact guarantee.

In [11]:
# Feature importance / relationship diagnostics (predictive interpretation, not causal proof)
if winner_name == "random_forest":
    # Tree-based importances after refit on train+val
    importances = winner.named_steps["model"].feature_importances_
    imp_df = pd.DataFrame({"feature": num_cols, "importance": importances}).sort_values("importance", ascending=False)
    display(imp_df.head(20))
    top_feature_rows = imp_df.head(5).to_dict(orient="records")
elif winner_name == "ridge":
    coefs = winner.named_steps["model"].coef_
    coef_df = pd.DataFrame({"feature": num_cols, "coefficient": coefs, "abs_coef": np.abs(coefs)}).sort_values("abs_coef", ascending=False)
    display(coef_df.head(20))
    top_feature_rows = coef_df.head(5).to_dict(orient="records")
else:
    top_feature_rows = []
    print("Baseline selected; no model-based feature importances.")

# Outlier sensitivity check: metrics excluding the single largest test month
if len(y_test) > 1:
    y_test_arr = np.array(y_test, dtype=float)
    pred_arr = np.array(test_pred, dtype=float)
    max_idx = int(np.argmax(y_test_arr))
    keep_mask = np.ones(len(y_test_arr), dtype=bool)
    keep_mask[max_idx] = False

    robust_mae = float(mean_absolute_error(y_test_arr[keep_mask], pred_arr[keep_mask]))
    robust_wape = float(np.sum(np.abs(y_test_arr[keep_mask] - pred_arr[keep_mask])) / np.sum(np.abs(y_test_arr[keep_mask])) * 100)

    print("\nOutlier sensitivity (excluding largest actual test month):")
    print("largest_actual_month_value:", round(float(y_test_arr[max_idx]), 2))
    print("mae_without_largest_month:", round(robust_mae, 2))
    print("wape_without_largest_month_pct:", round(robust_wape, 2))

print("\nTop features used by winner:")
for row in top_feature_rows:
    if "importance" in row:
        print(f"- {row['feature']}: importance={row['importance']:.4f}")
    else:
        print(f"- {row['feature']}: abs_coef={row['abs_coef']:.4f}")

,feature,importance
8,allocated_amount_total,0.240296
1,avg_gift_value,0.139387
10,month_num,0.122300
3,unique_supporters,0.063171
2,recurring_share,0.047857
4,active_supporter_share,0.047001
7,referred_post_avg_engagement,0.043560
27,channel_count_Event,0.041066
20,type_count_InKind,0.033620
15,lag_gift_count_1,0.027480


## 5. Causal and Relationship Analysis

This pipeline is **predictive**, so model relationships are interpreted as patterns that improve next-month forecasting, not as direct causal effects.

### What relationships were discovered and why they make sense
In the current run, the model beats the naive baseline on holdout (baseline MAE about **5695** vs model MAE about **3614**), indicating that month-level structure carries real predictive signal beyond "next month equals this month."

The most important features are typically:
- `allocated_amount_total`
- `rolling6_total_mean`
- `month_num` (seasonality)
- `avg_gift_value`
- `unique_supporters`

These are theoretically plausible for fundraising forecasting:
- Recent aggregate totals and rolling means reflect momentum/regime.
- Month-of-year captures seasonal campaign timing.
- Gift size and supporter breadth reflect both donor mix and engagement depth.

### Outliers and robustness
Production data can contain extreme months (for example, testing spikes). We handle this in several ways:
- Time-based validation (no shuffle) prevents leakage and gives realistic stress.
- Model complexity controls (`max_depth`, `min_samples_leaf`) reduce overreaction to single spikes.
- We report robust aggregate metrics (WAPE/sMAPE) instead of relying only on MAPE.
- We include an outlier sensitivity check by recomputing error with the largest test month removed, so we can see whether one extreme month dominates performance.

### Causal claims: what is and is not defensible
**Defensible:** these features are associated with better out-of-sample prediction and likely summarize donor behavior dynamics.

**Not defensible as causal:** we cannot claim that changing one feature (for example, boosting allocations) will itself cause the same change in next-month donations. Campaign strategy, economic context, and unobserved donor factors can confound observed relationships.

This demonstrates the prediction-vs-explanation distinction: we use relationships to forecast accurately, while explicitly avoiding causal claims without stronger identification methods.

## 6. Deployment Notes

### Artifact export
This notebook saves a deployment-ready model bundle as `.joblib` under `ml-pipelines/artifacts/`.

### Repo integration references (phase 2)
- ML service endpoint implementation location: `ml-service/app/main.py`
- Backend proxy/client location: `backend/Lighthouse.Web/Services/SocialMediaAnalyticsClient.cs` and `backend/Lighthouse.Web/Controllers/Api/*`
- Reports & analytics UI card location: `frontend/src/pages/AdminAnalytics.tsx`

### Production strategy for this forecast
- Use live DB data (same pattern as other ml-service endpoints), rebuild monthly features, load artifact, and return next-month forecast quickly.
- Trigger refresh on donation writes or run lightweight scheduled refresh to balance speed and accuracy.

In [12]:
# Save winner artifact for deployment
artifact_path = ART / "donation_prediction_next_month_model.joblib"

bundle = {
    "model_name": winner_name,
    "model": winner,
    "feature_columns": num_cols,
    "target": "target_next_month_total",
    "metrics_test": test_metrics,
    "training_window": {
        "first_month": str(model_df.iloc[0]["month"]),
        "last_month": str(model_df.iloc[val_end - 1]["month"]),
    },
    "notes": "Predicts next calendar month total estimated donation value from month-level lagged features.",
}

joblib.dump(bundle, artifact_path)
print("Saved artifact:", artifact_path)
print("Exists:", artifact_path.exists())

Saved artifact: /Users/lukemarchant/Desktop/IntextW2026/ml-pipelines/artifacts/donation_prediction_next_month_model.joblib
Exists: True


In [13]:
# Textbook crosswalk (chapters 0-17) used in this notebook
crosswalk = pd.DataFrame(
    [
        ("Ch 0-1", "End-to-end pipeline and business framing", "Sections 1 and 6 narrative"),
        ("Ch 2-3", "Data understanding and quality", "Missingness, outliers, and descriptive stats"),
        ("Ch 4-5", "Data preparation and leakage control", "Month-level features + next-month target shift"),
        ("Ch 7", "Reproducible pipeline", "Deterministic feature engineering and sklearn Pipeline"),
        ("Ch 8", "Automation mindset", "Single-run top-to-bottom reproducible notebook"),
        ("Ch 9-11", "Linear modeling and interpretation", "Ridge baseline and coefficient diagnostics"),
        ("Ch 12-14", "Tree/ensemble predictive modeling", "Random forest with tuning"),
        ("Ch 15", "Validation and model selection", "Temporal split + TimeSeries CV + untouched test"),
        ("Ch 16", "Feature selection and overfitting control", "Lag features, conservative columns, complexity controls"),
        ("Ch 17", "Deployment and monitoring", "Joblib artifact + integration notes"),
    ],
    columns=["chapter_group", "principle", "where_applied"],
)
display(crosswalk)

,chapter_group,principle,where_applied
0,Ch 0-1,End-to-end pipeline and business framing,Sections 1 and 6 narrative
1,Ch 2-3,Data understanding and quality,"Missingness, outliers, and descriptive stats"
2,Ch 4-5,Data preparation and leakage control,Month-level features + next-month target shift
3,Ch 7,Reproducible pipeline,Deterministic feature engineering and sklearn ...
4,Ch 8,Automation mindset,Single-run top-to-bottom reproducible notebook
5,Ch 9-11,Linear modeling and interpretation,Ridge baseline and coefficient diagnostics
6,Ch 12-14,Tree/ensemble predictive modeling,Random forest with tuning
7,Ch 15,Validation and model selection,Temporal split + TimeSeries CV + untouched test
8,Ch 16,Feature selection and overfitting control,"Lag features, conservative columns, complexity..."
9,Ch 17,Deployment and monitoring,Joblib artifact + integration notes
